## Predictive Modeling and Comparison
This notebook explores and evaluates different regression algorithms for predicting bicycle demand.

In [1]:
import pandas as pd
import numpy as np

# Data splitting and cross-validation
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold
)

# Pipeline and preprocessing
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
    SplineTransformer
)

# Regression models
from sklearn.linear_model import PoissonRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Evaluation metrics
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [2]:
# Load dataset
df = pd.read_csv("dataset_final.csv")

In [3]:
# Categorical features
nominal_cat = [
    "city_name",
    "weather_group",
    "working_day",
    "school_break" #,
    # "uni_break"
]

ordinal_cat = [
    "month",
    "hour"
]

# Numerical features
numerical = [
    "temp",
    "humidity",
    "wind_speed",
    "rain_1h",
    "snow_1h"
]

# Define X and y
X = df[nominal_cat + ordinal_cat + numerical]

y = df["total_bike_usage"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=31416
)

In [5]:
# One-Hot Encoder
oh_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Ordinal Encoder
ord_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

spline_encoder = SplineTransformer(
    n_knots=13,
    extrapolation="periodic"
)

# Scalers
scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

# Preprocessor 1: OneHot + Ordinal
preprocessor1 = make_column_transformer(
    (oh_encoder, nominal_cat),
    (ord_encoder, ordinal_cat)
)

# Preprocessor 2: OneHot + OneHot
preprocessor2 = make_column_transformer(
    (oh_encoder, nominal_cat),
    (oh_encoder, ordinal_cat)
)

# Preprocessor 3: OneHot + OneHot + StandardScaler
preprocessor3 = make_column_transformer(
    (oh_encoder, nominal_cat),
    (oh_encoder, ordinal_cat),
    (scaler, numerical)
)

# Preprocessor 4: OneHot + OneHot + MinMaxScaler
preprocessor4 = make_column_transformer(
    (oh_encoder, nominal_cat),
    (oh_encoder, ordinal_cat),
    (minmax_scaler, numerical)
)

# Preprocessor 5: OneHot + Ordinal + StandardScaler
preprocessor5 = make_column_transformer(
    (oh_encoder, nominal_cat),
    (ord_encoder, ordinal_cat),
    (scaler, numerical)
)

# Preprocessor 6: OneHot + Ordinal + MinMaxScaler
preprocessor6 = make_column_transformer(
    (oh_encoder, nominal_cat),
    (ord_encoder, ordinal_cat),
    (minmax_scaler, numerical)
)

# Preprocessor 7: Spline(hour) + OneHot(nominal) + Ordinal(month) + Numerical
preprocessor7 = make_column_transformer(
    (oh_encoder, nominal_cat),
    (spline_encoder, ["hour"]),
    (ord_encoder, ["month"]),
    (scaler, numerical)
)


# Preprocessor 8: Spline(month) + OneHot(nominal) + Ordinal(hour) + Numerical
preprocessor8 = make_column_transformer(
    (oh_encoder, nominal_cat),
    (spline_encoder, ["month"]),
    (ord_encoder, ["hour"]),
    (scaler, numerical)
)


# Preprocessor 9: Spline(hour, month) + OneHot(nominal) + Numerical
preprocessor9 = make_column_transformer(
    (oh_encoder, nominal_cat),
    (spline_encoder, ["hour", "month"]),
    (scaler, numerical)
)

## Model Evaluation Functions

In [6]:
def get_cv_fold_results(grid_search, model_name, cv):
    """
    Extracts training and validation metrics for each CV fold
    using the best hyperparameter combination from GridSearchCV.

    Calculates the mean across folds and returns the results
    as a DataFrame.
    """
    cv_results_df = pd.DataFrame(grid_search.cv_results_)

    best_idx = grid_search.best_index_

    n_splits = cv.get_n_splits()

    fold_details = pd.DataFrame({
        "Fold": [f"Fold {i+1}" for i in range(n_splits)],

        "Train RMSE": [
            -cv_results_df.loc[best_idx, f"split{i}_train_RMSE"]
            for i in range(n_splits)
        ],

        "Validation RMSE": [
            -cv_results_df.loc[best_idx, f"split{i}_test_RMSE"]
            for i in range(n_splits)
        ],

        "Train MAE": [
            -cv_results_df.loc[best_idx, f"split{i}_train_MAE"]
            for i in range(n_splits)
        ],

        "Validation MAE": [
            -cv_results_df.loc[best_idx, f"split{i}_test_MAE"]
            for i in range(n_splits)
        ],

        "Train R2": [
            cv_results_df.loc[best_idx, f"split{i}_train_R2"]
            for i in range(n_splits)
        ],

        "Validation R2": [
            cv_results_df.loc[best_idx, f"split{i}_test_R2"]
            for i in range(n_splits)
        ]
    })

    # Calculate mean across folds
    mean_row = fold_details.mean(numeric_only=True).to_frame().T
    mean_row.insert(0, "Fold", "Mean")

    # Combine results
    fold_details_final = pd.concat(
        [fold_details, mean_row],
        ignore_index=True
    )

    print(f"\n{model_name} - CV Fold Results:")
    print(fold_details_final.round(4).to_string(index=False))
    
    return fold_details_final

In [7]:
def evaluate_final_model(best_model, X_train, X_test, y_train, y_test):
    """
    Evaluates the best model on the training and test datasets.

    Calculates RMSE, MAE, and R2 scores and returns
    the evaluation results as a DataFrame.
    """
    # Final training predictions
    y_train_pred = best_model.predict(X_train)

    # Final test predictions
    y_test_pred = best_model.predict(X_test)

    # Final evaluation
    final_results = pd.DataFrame({
        "Dataset": ["Final Training", "Final Test"],

        "RMSE": [
            np.sqrt(mean_squared_error(y_train, y_train_pred)),
            np.sqrt(mean_squared_error(y_test, y_test_pred))
        ],

        "MAE": [
            mean_absolute_error(y_train, y_train_pred),
            mean_absolute_error(y_test, y_test_pred)
        ],

        "R2": [
            r2_score(y_train, y_train_pred),
            r2_score(y_test, y_test_pred)
        ]
    })

    print("\nFinal Model Results:")
    print(final_results.round(4).to_string(index=False))

    return final_results

In [8]:
def get_model_summary(model_name, grid_search, final_results):
    """
    Creates a summary of the model's best CV and final test results.
    """

    model_summary = {
        "Model": model_name,
        "Best CV RMSE": -grid_search.best_score_,
        "Test RMSE": final_results.loc[1, "RMSE"],
        "Test MAE": final_results.loc[1, "MAE"],
        "Test R2": final_results.loc[1, "R2"],
        "Best Parameters": grid_search.best_params_
    }

    return model_summary

## HistGradientBoosting Regressor

In [9]:
# Define pipeline
hist_pipe = Pipeline([
    ("preprocessor", preprocessor1),
    ("regressor", HistGradientBoostingRegressor(
        loss="poisson",
        random_state=42
    ))
])

# Define parameter grid
hist_param_grid = {
    "preprocessor": [
        preprocessor1,
        preprocessor2,
        preprocessor3,
        preprocessor4,
        preprocessor5,
        preprocessor6,
        preprocessor7,
        preprocessor8,
        preprocessor9
    ],
    "regressor__max_iter": [100, 200, 300],
    "regressor__learning_rate": [0.05, 0.1],
    "regressor__max_leaf_nodes": [10, 15, 20]
}

# Cross-validation strategy
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation metrics
scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

# Grid Search
hist_grid_search = GridSearchCV(
    estimator=hist_pipe,
    param_grid=hist_param_grid,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

hist_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 162 candidates, totalling 810 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'preprocessor': [ColumnTransfo...h', 'hour'])]), ColumnTransfo...h', 'hour'])]), ...], 'regressor__learning_rate': [0.05, 0.1], 'regressor__max_iter': [100, 200, ...], 'regressor__max_leaf_nodes': [10, 15, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'MAE': 'neg_mean_absolute_error', 'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'RMSE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_

In [10]:
print("Best Parameters:")
print(hist_grid_search.best_params_)

print("\nBest CV RMSE:")
print(-hist_grid_search.best_score_)

Best Parameters:
{'preprocessor': ColumnTransformer(transformers=[('onehotencoder',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['city_name', 'weather_group', 'working_day',
                                  'school_break']),
                                ('ordinalencoder',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['month', 'hour']),
                                ('standardscaler', StandardScaler(),
                                 ['temp', 'humidity', 'wind_speed', 'rain_1h',
                                  'snow_1h'])]), 'regressor__learning_rate': 0.1, 'regressor__max_iter': 300, 'regressor__max_leaf_nodes': 20}

Best CV RMSE:
50.01177537807028


In [11]:
# CV fold results
hist_fold_details_final = get_cv_fold_results(
    hist_grid_search,
    "HistGradientBoosting",
    cv
)


HistGradientBoosting - CV Fold Results:
  Fold  Train RMSE  Validation RMSE  Train MAE  Validation MAE  Train R2  Validation R2
Fold 1     41.3098          51.3348    26.7599         32.5545    0.9509         0.9225
Fold 2     41.0697          49.1765    27.0009         32.1664    0.9511         0.9313
Fold 3     41.7005          52.4728    27.3685         33.0589    0.9490         0.9249
Fold 4     42.3840          48.5209    27.5992         30.9531    0.9485         0.9295
Fold 5     42.2594          48.5539    27.1916         31.2236    0.9487         0.9302
  Mean     41.7447          50.0118    27.1840         31.9913    0.9497         0.9277


In [12]:
# Final model evaluation
hist_best_model = hist_grid_search.best_estimator_

hist_final_results = evaluate_final_model(
    hist_best_model,
    X_train,
    X_test,
    y_train,
    y_test
)


Final Model Results:
       Dataset    RMSE     MAE     R2
Final Training 42.8556 27.7845 0.9469
    Final Test 51.2254 32.7111 0.9290


## LGBM Regressor

In [13]:
# Define LightGBM pipeline
lgbm_pipe = Pipeline([
    ("preprocessor", preprocessor1),
    ("regressor", LGBMRegressor(
        objective="regression",
        random_state=42,
        verbosity=-1
    ))
])

# Define parameter grid
lgbm_param_grid = {
    "preprocessor": [
        preprocessor1,
        preprocessor2,
        preprocessor3,
        preprocessor4,
        preprocessor5,
        preprocessor6,
        preprocessor7,
        preprocessor8,
        preprocessor9
    ],
    "regressor__n_estimators": [200, 500],
    "regressor__learning_rate": [0.03, 0.1],
    "regressor__num_leaves": [15, 31],
    "regressor__max_depth": [-1, 7]
}

# Cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation metrics
scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

# Grid Search
lgbm_grid_search = GridSearchCV(
    estimator=lgbm_pipe,
    param_grid=lgbm_param_grid,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

lgbm_grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 144 candidates, totalling 720 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...rbosity=-1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'preprocessor': [ColumnTransfo...h', 'hour'])]), ColumnTransfo...h', 'hour'])]), ...], 'regressor__learning_rate': [0.03, 0.1], 'regressor__max_depth': [-1, 7], 'regressor__n_estimators': [200, 500], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'MAE': 'neg_mean_absolute_error', 'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'RMSE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validat

In [14]:
print("Best Parameters:")
print(lgbm_grid_search.best_params_)

print("\nBest CV RMSE:")
print(-lgbm_grid_search.best_score_)

Best Parameters:
{'preprocessor': ColumnTransformer(transformers=[('onehotencoder',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['city_name', 'weather_group', 'working_day',
                                  'school_break']),
                                ('splinetransformer',
                                 SplineTransformer(extrapolation='periodic',
                                                   n_knots=13),
                                 ['month']),
                                ('ordinalencoder',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['hour']),
                                ('standardscaler', StandardScaler(),
                                 ['temp', 'humidity', 'wind_speed', 'rain_1h',
          

In [15]:
# CV fold results
lgbm_fold_details_final = get_cv_fold_results(
    lgbm_grid_search,
    "LightGBM",
    cv
)


LightGBM - CV Fold Results:
  Fold  Train RMSE  Validation RMSE  Train MAE  Validation MAE  Train R2  Validation R2
Fold 1     31.1293          49.3329    21.2719         32.2105    0.9721         0.9285
Fold 2     31.9687          47.0965    21.5793         31.3301    0.9704         0.9370
Fold 3     31.2954          51.1688    21.2655         32.8590    0.9713         0.9286
Fold 4     31.8073          46.7020    21.6297         30.5769    0.9710         0.9347
Fold 5     31.4257          47.9042    21.4345         31.0606    0.9716         0.9320
  Mean     31.5253          48.4409    21.4362         31.6074    0.9713         0.9322


In [16]:
# Final model evaluation
lgbm_best_model = lgbm_grid_search.best_estimator_

lgbm_final_results = evaluate_final_model(
    lgbm_best_model,
    X_train,
    X_test,
    y_train,
    y_test
)


Final Model Results:
       Dataset    RMSE     MAE     R2
Final Training 33.3266 22.5242 0.9679
    Final Test 49.4710 31.9979 0.9338


## XGB Regressor

In [17]:
# Define XGBoost pipeline
xgb_pipe = Pipeline([
    ("preprocessor", preprocessor1),
    ("regressor", XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=1
    ))
])

# Define parameter grid
xgb_param_grid = {
    "preprocessor": [
        preprocessor1,
        preprocessor2,
        preprocessor3,
        preprocessor4,
        preprocessor5,
        preprocessor6,
        preprocessor7,
        preprocessor8,
        preprocessor9
    ],
    "regressor__n_estimators": [200, 500],
    "regressor__learning_rate": [0.05, 0.1],
    "regressor__max_depth": [3, 5],
    "regressor__subsample": [0.8]
}

# Cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation metrics
scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

# Grid Search
xgb_grid_search = GridSearchCV(
    estimator=xgb_pipe,
    param_grid=xgb_param_grid,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

xgb_grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 72 candidates, totalling 360 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'preprocessor': [ColumnTransfo...h', 'hour'])]), ColumnTransfo...h', 'hour'])]), ...], 'regressor__learning_rate': [0.05, 0.1], 'regressor__max_depth': [3, 5], 'regressor__n_estimators': [200, 500], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'MAE': 'neg_mean_absolute_error', 'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'RMSE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_valida

In [18]:
print("Best Parameters:")
print(xgb_grid_search.best_params_)

print("\nBest CV RMSE:")
print(-xgb_grid_search.best_score_)

Best Parameters:
{'preprocessor': ColumnTransformer(transformers=[('onehotencoder',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['city_name', 'weather_group', 'working_day',
                                  'school_break']),
                                ('splinetransformer',
                                 SplineTransformer(extrapolation='periodic',
                                                   n_knots=13),
                                 ['month']),
                                ('ordinalencoder',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['hour']),
                                ('standardscaler', StandardScaler(),
                                 ['temp', 'humidity', 'wind_speed', 'rain_1h',
          

In [19]:
# CV fold results
xgb_fold_details_final = get_cv_fold_results(
    xgb_grid_search,
    "XGBoost",
    cv
)


XGBoost - CV Fold Results:
  Fold  Train RMSE  Validation RMSE  Train MAE  Validation MAE  Train R2  Validation R2
Fold 1     35.2026          51.8803    24.2152         33.9185    0.9644         0.9209
Fold 2     35.4642          48.8395    24.2632         32.7001    0.9635         0.9322
Fold 3     34.7079          53.0156    23.8056         34.2020    0.9647         0.9233
Fold 4     35.9453          48.8550    24.3635         32.4630    0.9630         0.9286
Fold 5     35.3443          50.5220    24.2723         33.2636    0.9641         0.9244
  Mean     35.3328          50.6225    24.1840         33.3094    0.9639         0.9259


In [20]:
# Final model evaluation
xgb_best_model = xgb_grid_search.best_estimator_

xgb_final_results = evaluate_final_model(
    xgb_best_model,
    X_train,
    X_test,
    y_train,
    y_test
)


Final Model Results:
       Dataset    RMSE     MAE     R2
Final Training 37.1176 25.2286 0.9602
    Final Test 51.8320 33.8360 0.9273


## CatBoost Regressor

In [21]:
# Define CatBoost pipeline
catboost_pipe = Pipeline([
    ("preprocessor", preprocessor1),
    ("regressor", CatBoostRegressor(
        loss_function="RMSE",
        random_seed=42,
        verbose=0,
        allow_writing_files=False
    ))
])

# Define parameter grid
catboost_param_grid = {
    "preprocessor": [
        preprocessor1,
        preprocessor2,
        preprocessor3,
        preprocessor4,
        preprocessor5,
        preprocessor6,
        preprocessor7,
        preprocessor8,
        preprocessor9
    ],

    "regressor__iterations": [200, 500],
    "regressor__learning_rate": [0.03, 0.1],
    "regressor__depth": [4, 6, 8],
    "regressor__l2_leaf_reg": [3, 5]
}

# Cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluation metrics
scoring = {
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2"
}

# Grid Search
catboost_grid_search = GridSearchCV(
    estimator=catboost_pipe,
    param_grid=catboost_param_grid,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

catboost_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... verbose=0))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'preprocessor': [ColumnTransfo...h', 'hour'])]), ColumnTransfo...h', 'hour'])]), ...], 'regressor__depth': [4, 6, ...], 'regressor__iterations': [200, 500], 'regressor__l2_leaf_reg': [3, 5], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'MAE': 'neg_mean_absolute_error', 'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'RMSE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for

In [22]:
print("Best Parameters:")
print(catboost_grid_search.best_params_)

print("\nBest CV RMSE:")
print(-catboost_grid_search.best_score_)

Best Parameters:
{'preprocessor': ColumnTransformer(transformers=[('onehotencoder',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['city_name', 'weather_group', 'working_day',
                                  'school_break']),
                                ('splinetransformer',
                                 SplineTransformer(extrapolation='periodic',
                                                   n_knots=13),
                                 ['month']),
                                ('ordinalencoder',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['hour']),
                                ('standardscaler', StandardScaler(),
                                 ['temp', 'humidity', 'wind_speed', 'rain_1h',
          

In [23]:
# CV fold results
catboost_fold_details_final = get_cv_fold_results(
    catboost_grid_search,
    "CatBoost",
    cv
)


CatBoost - CV Fold Results:
  Fold  Train RMSE  Validation RMSE  Train MAE  Validation MAE  Train R2  Validation R2
Fold 1     35.0907          49.3519    23.8414         31.7600    0.9646         0.9284
Fold 2     36.2544          45.9289    24.4168         30.8758    0.9619         0.9401
Fold 3     35.0789          50.4436    23.7579         32.0428    0.9639         0.9306
Fold 4     35.7858          45.2454    24.0819         29.7071    0.9633         0.9387
Fold 5     35.9145          47.3360    24.2721         30.6279    0.9630         0.9336
  Mean     35.6248          47.6612    24.0740         31.0027    0.9633         0.9343


In [24]:
# Final model evaluation
catboost_best_model = catboost_grid_search.best_estimator_

catboost_final_results = evaluate_final_model(
    catboost_best_model,
    X_train,
    X_test,
    y_train,
    y_test
)


Final Model Results:
       Dataset    RMSE     MAE     R2
Final Training 36.9704 24.8768 0.9605
    Final Test 48.0455 31.2094 0.9375


## Random Forest Regressor

In [25]:
# Define Random Forest pipeline
rf_pipe = Pipeline([
    ("preprocessor", preprocessor1),
    ("regressor", RandomForestRegressor(
        random_state=42,
        n_jobs=1
    ))
])

# Define parameter grid
rf_param_grid = {
    "preprocessor": [
        preprocessor1,
        preprocessor2,
        preprocessor3,
        preprocessor4,
        preprocessor5,
        preprocessor6,
        preprocessor7,
        preprocessor8,
        preprocessor9        
    ],

    "regressor__n_estimators": [200, 350 ],
    "regressor__max_depth": [None, 20],
    "regressor__min_samples_split": [5, 8],
    "regressor__min_samples_leaf": [3, 5]
}

# Cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Grid Search
rf_grid_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=rf_param_grid,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

rf_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'preprocessor': [ColumnTransfo...h', 'hour'])]), ColumnTransfo...h', 'hour'])]), ...], 'regressor__max_depth': [None, 20], 'regressor__min_samples_leaf': [3, 5], 'regressor__min_samples_split': [5, 8], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'MAE': 'neg_mean_absolute_error', 'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'RMSE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_valid

In [26]:
print("Best Parameters:")
print(rf_grid_search.best_params_)

print("\nBest CV RMSE:")
print(-rf_grid_search.best_score_)

Best Parameters:
{'preprocessor': ColumnTransformer(transformers=[('onehotencoder',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['city_name', 'weather_group', 'working_day',
                                  'school_break']),
                                ('splinetransformer',
                                 SplineTransformer(extrapolation='periodic',
                                                   n_knots=13),
                                 ['month']),
                                ('ordinalencoder',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['hour']),
                                ('standardscaler', StandardScaler(),
                                 ['temp', 'humidity', 'wind_speed', 'rain_1h',
          

In [27]:
# CV fold results
rf_fold_details_final = get_cv_fold_results(
    rf_grid_search,
    "RandomForest",
    cv
)


RandomForest - CV Fold Results:
  Fold  Train RMSE  Validation RMSE  Train MAE  Validation MAE  Train R2  Validation R2
Fold 1     31.6516          53.3772    19.9406         34.3643    0.9712         0.9163
Fold 2     31.9787          52.1605    19.9866         34.0897    0.9703         0.9227
Fold 3     31.5275          54.5355    19.8848         34.4857    0.9709         0.9189
Fold 4     31.8076          50.7108    19.9607         32.8070    0.9710         0.9230
Fold 5     31.7136          50.2265    19.9387         32.4641    0.9711         0.9253
  Mean     31.7358          52.2021    19.9423         33.6422    0.9709         0.9212


In [28]:
# Final model evaluation
rf_best_model = rf_grid_search.best_estimator_

rf_final_results = evaluate_final_model(
    rf_best_model,
    X_train,
    X_test,
    y_train,
    y_test
)


Final Model Results:
       Dataset    RMSE     MAE     R2
Final Training 30.8013 19.3553 0.9726
    Final Test 50.6738 32.6436 0.9305


## KNeighbors Regressor

In [29]:
# Define KNN pipeline
knn_pipe = Pipeline([
    ("preprocessor", preprocessor1),
    ("regressor", KNeighborsRegressor(n_jobs=1))
])

# Define parameter grid
knn_param_grid = {
    "preprocessor": [
        preprocessor1,
        preprocessor2,
        preprocessor3,
        preprocessor4,
        preprocessor5,
        preprocessor6,
        preprocessor7,
        preprocessor8,
        preprocessor9
    ],
    "regressor__n_neighbors": [3, 5, 10, 15],
    "regressor__weights": ["uniform", "distance"],
    "regressor__p": [1, 2]
}

# Cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Grid Search
knn_grid_search = GridSearchCV(
    estimator=knn_pipe,
    param_grid=knn_param_grid,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

knn_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 144 candidates, totalling 720 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...r(n_jobs=1))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'preprocessor': [ColumnTransfo...h', 'hour'])]), ColumnTransfo...h', 'hour'])]), ...], 'regressor__n_neighbors': [3, 5, ...], 'regressor__p': [1, 2], 'regressor__weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'MAE': 'neg_mean_absolute_error', 'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'RMSE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` fo

In [30]:
print("Best Parameters:")
print(knn_grid_search.best_params_)

print("\nBest CV RMSE:")
print(-knn_grid_search.best_score_)

Best Parameters:
{'preprocessor': ColumnTransformer(transformers=[('onehotencoder',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['city_name', 'weather_group', 'working_day',
                                  'school_break']),
                                ('ordinalencoder',
                                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['month', 'hour']),
                                ('standardscaler', StandardScaler(),
                                 ['temp', 'humidity', 'wind_speed', 'rain_1h',
                                  'snow_1h'])]), 'regressor__n_neighbors': 10, 'regressor__p': 1, 'regressor__weights': 'distance'}

Best CV RMSE:
56.72218981379211


In [31]:
# CV-fold results
knn_fold_details_final = get_cv_fold_results(
    knn_grid_search,
    "KNN",
    cv
)


KNN - CV Fold Results:
  Fold  Train RMSE  Validation RMSE  Train MAE  Validation MAE  Train R2  Validation R2
Fold 1         0.0          59.7347        0.0         39.0160       1.0         0.8951
Fold 2         0.0          55.7495        0.0         37.0910       1.0         0.9117
Fold 3         0.0          58.6593        0.0         37.9007       1.0         0.9061
Fold 4         0.0          53.5929        0.0         35.2912       1.0         0.9140
Fold 5         0.0          55.8745        0.0         36.2506       1.0         0.9075
  Mean         0.0          56.7222        0.0         37.1099       1.0         0.9069


In [32]:
# Final model evaluation
knn_best_model = knn_grid_search.best_estimator_

knn_final_results = evaluate_final_model(
    knn_best_model,
    X_train,
    X_test,
    y_train,
    y_test
)


Final Model Results:
       Dataset    RMSE     MAE     R2
Final Training  0.0000  0.0000 1.0000
    Final Test 55.0195 36.0531 0.9181


## Poisson Regressor

In [33]:
# Define Poisson pipeline
poisson_pipe = Pipeline([
    ("preprocessor", preprocessor1),
    ("regressor", PoissonRegressor(max_iter=1000))
])

# Define parameter grid
poisson_param_grid = {
    "preprocessor": [
        preprocessor1,
        preprocessor2,
        preprocessor3,
        preprocessor4,
        preprocessor5,
        preprocessor6,
        preprocessor7,
        preprocessor8,
        preprocessor9
    ],
    "regressor__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]
}

# Cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Grid Search
poisson_grid_search = GridSearchCV(
    estimator=poisson_pipe,
    param_grid=poisson_param_grid,
    scoring=scoring,
    refit="RMSE",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

poisson_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 45 candidates, totalling 225 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=1000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'preprocessor': [ColumnTransfo...h', 'hour'])]), ColumnTransfo...h', 'hour'])]), ...], 'regressor__alpha': [0.001, 0.01, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'MAE': 'neg_mean_absolute_error', 'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'RMSE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... vers

In [34]:
print("Best Parameters:")
print(poisson_grid_search.best_params_)

print("\nBest CV RMSE:")
print(-poisson_grid_search.best_score_)

Best Parameters:
{'preprocessor': ColumnTransformer(transformers=[('onehotencoder-1',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['city_name', 'weather_group', 'working_day',
                                  'school_break']),
                                ('onehotencoder-2',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['month', 'hour']),
                                ('minmaxscaler', MinMaxScaler(),
                                 ['temp', 'humidity', 'wind_speed', 'rain_1h',
                                  'snow_1h'])]), 'regressor__alpha': 0.001}

Best CV RMSE:
78.78851886210236


In [35]:
poisson_fold_details_final = get_cv_fold_results(
    poisson_grid_search,
    "Poisson Regression",
    cv
)


Poisson Regression - CV Fold Results:
  Fold  Train RMSE  Validation RMSE  Train MAE  Validation MAE  Train R2  Validation R2
Fold 1     78.4308          79.0314    54.2984         54.9361    0.8231         0.8164
Fold 2     78.0860          79.2491    54.0579         55.0894    0.8231         0.8215
Fold 3     77.6073          83.0719    54.0596         56.2108    0.8234         0.8117
Fold 4     78.9773          75.6562    54.7203         52.5008    0.8214         0.8287
Fold 5     78.8198          76.9340    54.3923         54.0683    0.8216         0.8247
  Mean     78.3843          78.7885    54.3057         54.5611    0.8225         0.8206


In [36]:
poisson_best_model = poisson_grid_search.best_estimator_

poisson_final_results = evaluate_final_model(
    poisson_best_model,
    X_train,
    X_test,
    y_train,
    y_test
)


Final Model Results:
       Dataset    RMSE     MAE     R2
Final Training 78.4269 54.3259 0.8223
    Final Test 78.5968 54.4486 0.8328


## Model Comparisons

In [37]:
all_model_results = []

# HistGradientBoosting
all_model_results.append(
    get_model_summary("HistGradientBoosting", hist_grid_search, hist_final_results)
)

# LightGBM
all_model_results.append(
    get_model_summary("LightGBM", lgbm_grid_search, lgbm_final_results)
)

# XGBoost
all_model_results.append(
    get_model_summary("XGBoost", xgb_grid_search, xgb_final_results)
)

# CatBoost
all_model_results.append(
    get_model_summary("CatBoost", catboost_grid_search, catboost_final_results)
)

# Random Forest
all_model_results.append(
    get_model_summary("Random Forest", rf_grid_search, rf_final_results)
)

# KNN
all_model_results.append(
    get_model_summary("KNN", knn_grid_search, knn_final_results)
)

# Poisson
all_model_results.append(
    get_model_summary("Poisson Regression", poisson_grid_search, poisson_final_results)
)

In [38]:
# Create model comparison table
model_comparison = pd.DataFrame(all_model_results)

model_comparison = model_comparison.sort_values(
    by="Test RMSE",
    ascending=True
).reset_index(drop=True)

model_comparison

,Model,Best CV RMSE,Test RMSE,Test MAE,Test R2,Best Parameters
0,CatBoost,47.661157,48.045538,31.209419,0.937524,{'preprocessor': ColumnTransformer(transformer...
1,LightGBM,48.440904,49.471035,31.997908,0.933762,{'preprocessor': ColumnTransformer(transformer...
2,Random Forest,52.202096,50.673756,32.643553,0.930502,{'preprocessor': ColumnTransformer(transformer...
3,HistGradientBoosting,50.011775,51.225410,32.711145,0.928981,{'preprocessor': ColumnTransformer(transformer...
4,XGBoost,50.622481,51.831963,33.835992,0.927289,{'preprocessor': ColumnTransformer(transformer...
5,KNN,56.722190,55.019532,36.053087,0.918071,{'preprocessor': ColumnTransformer(transformer...
6,Poisson Regression,78.788519,78.596758,54.448636,0.832808,{'preprocessor': ColumnTransformer(transformer...


In [39]:
# import pickle

# # Save the best model
# with open("catboost_model_spline.pkl", "wb") as f:
#     pickle.dump(catboost_grid_search.best_estimator_, f)